notebook based on

Andrij https://www.kaggle.com/code/aikhmelnytskyy/public-krni-pdi-with-two-additional-models

SAMUEL https://www.kaggle.com/code/muelsamu/simple-tabpfn-approach-for-score-of-15-in-1-min

Vaibhav Jain https://www.kaggle.com/code/vaibhavjain2004/public-krni-pdi?scriptVersionId=133524570

Vadim Kamaev https://www.kaggle.com/code/vadimkamaev/postprocessin-ensemble

In [ ]:
!pip install tabpfn --no-index --find-links=file:///kaggle/input/pip-packages-icr/pip-packages
!mkdir -p /opt/conda/lib/python3.10/site-packages/tabpfn/models_diff
!cp /kaggle/input/pip-packages-icr/pip-packages/prior_diff_real_checkpoint_n_0_epoch_100.cpkt /opt/conda/lib/python3.10/site-packages/tabpfn/models_diff/

In [ ]:
import math
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder,normalize
from sklearn.ensemble import GradientBoostingClassifier,RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
import imblearn
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import xgboost
import inspect
from collections import defaultdict
from tabpfn import TabPFNClassifier
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv')
test = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv')
sample = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/sample_submission.csv')
greeks = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')

In [ ]:
# lb = LabelEncoder()
# train['EJ'] = lb.fit_transform(train['EJ']).astype(float)
# test['EJ'] = lb.fit_transform(test['EJ']).astype(float)

In [ ]:
first_category = train.EJ.unique()[0]
train.EJ = train.EJ.eq(first_category).astype('int')
test.EJ = test.EJ.eq(first_category).astype('int')

In [ ]:
def random_under_sampler(df):
    # Calculate the number of samples for each label. 
    neg, pos = np.bincount(df['Class'])

    # Choose the samples with class label `1`.
    one_df = df.loc[df['Class'] == 1] 
    # Choose the samples with class label `0`.
    zero_df = df.loc[df['Class'] == 0]
    # Select `pos` number of negative samples.
    # This makes sure that we have equal number of samples for each label.
    zero_df = zero_df.sample(n=pos)

    # Join both label dataframes.
    undersampled_df = pd.concat([zero_df, one_df])

    # Shuffle the data and return
    return undersampled_df.sample(frac = 1)

In [ ]:
train_good = random_under_sampler(train)

In [ ]:
train_good.shape

In [ ]:
predictor_columns = [n for n in train.columns if n != 'Class' and n != 'Id']
x= train[predictor_columns]
y = train['Class']

In [ ]:
# x_norm = np.array(x_norm)
# y_ros = np.array(y_ros)

In [ ]:
from sklearn.model_selection import KFold as KF, GridSearchCV
cv_outer = KF(n_splits = 10, shuffle=True, random_state=42)
cv_inner = KF(n_splits = 5, shuffle=True, random_state=42)

In [ ]:
def balanced_log_loss(y_true, y_pred):
    # y_true: correct labels 0, 1
    # y_pred: predicted probabilities of class=1
    # calculate the number of observations for each class
    N_0 = np.sum(1 - y_true)
    N_1 = np.sum(y_true)
    # calculate the weights for each class to balance classes
    w_0 = 1 / N_0
    w_1 = 1 / N_1
    # calculate the predicted probabilities for each class
    p_1 = np.clip(y_pred, 1e-15, 1 - 1e-15)
    p_0 = 1 - p_1
    # calculate the summed log loss for each class
    log_loss_0 = -np.sum((1 - y_true) * np.log(p_0))
    log_loss_1 = -np.sum(y_true * np.log(p_1))
    # calculate the weighted summed logarithmic loss
    # (factgor of 2 included to give same result as LL with balanced input)
    balanced_log_loss = 2*(w_0 * log_loss_0 + w_1 * log_loss_1) / (w_0 + w_1)
    # return the average log loss
    return balanced_log_loss/(N_0+N_1)

In [ ]:
class Ensemble():
    def __init__(self):
        self.imputer = SimpleImputer(missing_values=np.nan, strategy='median')

        self.classifiers =[xgboost.XGBClassifier(n_estimators=100,max_depth=3,learning_rate=0.2,subsample=0.9,colsample_bytree=0.85),
                           xgboost.XGBClassifier(),
                           TabPFNClassifier(N_ensemble_configurations=24),
                          TabPFNClassifier(N_ensemble_configurations=64)]
    
    def fit(self,X,y):
        y = y.values
        unique_classes, y = np.unique(y, return_inverse=True)
        self.classes_ = unique_classes
        first_category = X.EJ.unique()[0]
        X.EJ = X.EJ.eq(first_category).astype('int')
        X = self.imputer.fit_transform(X)
#         X = normalize(X,axis=0)
        for classifier in self.classifiers:
            if classifier==self.classifiers[2] or classifier==self.classifiers[3]:
                classifier.fit(X,y,overwrite_warning =True)
            else :
                classifier.fit(X, y)
     
    def predict_proba(self, x):
        x = self.imputer.transform(x)
#         x = normalize(x,axis=0)
        probabilities = np.stack([classifier.predict_proba(x) for classifier in self.classifiers])
        averaged_probabilities = np.mean(probabilities, axis=0)
        class_0_est_instances = averaged_probabilities[:, 0].sum()
        others_est_instances = averaged_probabilities[:, 1:].sum()
        # Weighted probabilities based on class imbalance
        new_probabilities = averaged_probabilities * np.array([[1/(class_0_est_instances if i==0 else others_est_instances) for i in range(averaged_probabilities.shape[1])]])
        return new_probabilities / np.sum(new_probabilities, axis=1, keepdims=1) 

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
def training(model, x,y,y_meta):
    outer_results = list()
    best_loss = np.inf
    split = 0
    splits = 5
    for train_idx,val_idx in tqdm(cv_inner.split(x), total = splits):
        split+=1
        x_train, x_val = x.iloc[train_idx],x.iloc[val_idx]
        y_train, y_val = y_meta.iloc[train_idx], y.iloc[val_idx]
                
        model.fit(x_train, y_train)
        y_pred = model.predict_proba(x_val)
        probabilities = np.concatenate((y_pred[:,:1], np.sum(y_pred[:,1:], 1, keepdims=True)), axis=1)
        p0 = probabilities[:,:1]
        p0[p0 > 0.86] = 1
        p0[p0 < 0.14] = 0
        y_p = np.empty((y_pred.shape[0],))
        for i in range(y_pred.shape[0]):
            if p0[i]>=0.5:
                y_p[i]= False
            else :
                y_p[i]=True
        y_p = y_p.astype(int)
        loss = balanced_log_loss(y_val,y_p)

        if loss<best_loss:
            best_model = model
            best_loss = loss
            print('best_model_saved')
        outer_results.append(loss)
        print('>val_loss=%.5f, split = %.1f' % (loss,split))
    print('LOSS: %.5f' % (np.mean(outer_results)))
    return best_model
    

In [ ]:
from datetime import datetime
times = greeks.Epsilon.copy()
times[greeks.Epsilon != 'Unknown'] = greeks.Epsilon[greeks.Epsilon != 'Unknown'].map(lambda x: datetime.strptime(x,'%m/%d/%Y').toordinal())
times[greeks.Epsilon == 'Unknown'] = np.nan

In [ ]:
train_pred_and_time = pd.concat((train, times), axis=1)
test_predictors = test[predictor_columns]
first_category = test_predictors.EJ.unique()[0]
test_predictors.EJ = test_predictors.EJ.eq(first_category).astype('int')
test_pred_and_time = np.concatenate((test_predictors, np.zeros((len(test_predictors), 1)) + train_pred_and_time.Epsilon.max() + 1), axis=1)

In [ ]:
ros = RandomOverSampler(random_state=42)

train_ros, y_ros = ros.fit_resample(train_pred_and_time, greeks.Alpha)
print('Original dataset shape')
print(greeks.Alpha.value_counts())
print('Resample dataset shape')
print( y_ros.value_counts())

In [ ]:
x_ros = train_ros.drop(['Class', 'Id'],axis=1)
y_ = train_ros.Class

In [ ]:
yt = Ensemble()

In [ ]:
m = training(yt,x_ros,y_,y_ros)

In [ ]:
y_.value_counts()/y_.shape[0]

In [ ]:
y_pred = m.predict_proba(test_pred_and_time)
probabilities = np.concatenate((y_pred[:,:1], np.sum(y_pred[:,1:], 1, keepdims=True)), axis=1)
p0 = probabilities[:,:1]
p0[p0 > 0.59] = 1 
p0[p0 < 0.28] = 0

In [ ]:
submission = pd.DataFrame(test["Id"], columns=["Id"])
submission["class_0"] = p0
submission["class_1"] = 1 - p0
# submission.to_csv('submission.csv', index=False)

In [ ]:
# submission_df = pd.read_csv('submission.csv')
# submission_df

# LB probe

Main idea: we know from [this discussion](https://www.kaggle.com/competitions/icr-identify-age-related-conditions/discussion/425621) that there are 26 objects of class 1 on LB and ~142 objects of class 0. That knowledge may help to determine which objects are belong to LB and which labels they have.

- sort predictions of model according to their probability (from highest to lowest)
- this model has high LB score => almost all predictions on LB are made correctly and all predictions in the beginnig of this sorted predicton array will have class 1 on LB
- select first 6 predictions and set them to special values that were selected to prevent LB score collisions
- set the rest predictions to 0.5
- submit prediction

If result is 0.69, that means that there are no LB objects of class 1 in first 7 prediction. If result is 0.7, that means that first object is in LB and has class 1 and the rest 5 objects are not in LB. If result is 0.75, that means that first two objects are in LB and have class 1, and the rest 4 objects are not in LB etc. Full list of object combinations and scores is below.

When we find, which of first 6 objects are belong to LB, we move to next 6 objects by setting *offset* variable to 6, etc.

When we find all 26 objects of class 1, we can repeat the same with objects of class 0. Need to sort our model predictions from lowest to highest for that.

To compeletely process all 400 obejcts we will need ~ 66 submission => it's impossible to do it from one account, need collective work. I tried to increase the number of objects per submission but this leads to collisions, 6 is the limit.

For objects of class 0 limit is 5 bit (and still there are some collisions), maybe this is because score is divided by bigger number of objects (~142 instead of 26).

In [ ]:
import math
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 500)

# set special values and offset
tp = [3e-1, 4e-2, 6e-4, 3e-7, 1e-12, 3e-13]

def balanced_log_loss(y_true, y_pred):
    # Nc is the number of observations
    N_1 = np.sum(y_true == 1, axis=0)
    N_0 = np.sum(y_true == 0, axis=0)

    N_inv_0 = 1/N_0 if N_0 > 0 else 0
    N_inv_1 = 1/N_1 if N_1 > 0 else 0

    # In order to avoid the extremes of the log function, each predicted probability 𝑝 is replaced with max(min(𝑝,1−10−15),10−15)
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)

    # balanced logarithmic loss
    loss_numerator = - N_inv_0 * np.sum((1 - y_true) * np.log(1 - y_pred)) - N_inv_1 * np.sum(y_true * np.log(y_pred))

    return loss_numerator / 2

# function that returns list of all object combinations and corresponding LB score
def get_score_list(tp, offset):
    score_dict = dict()

    y = np.array([1] * 26 + [0] * 142)
#     y = np.array([0] * 142 + [1] * 26) # change to this to find objects of class 0
    preds = [0.5] * 168

    for i in range(2**len(tp)):
        b = bin(i)
        b = '0' * (len(tp) - len(str(b[2:]))) + str(b)[2:]
        b = b[::-1]
        bl = [int(j) for j in b]
      
        for j, p in enumerate(bl):
            if p == 1:
                preds[j + offset] = tp[j]
            else:
                preds[j + offset] = 0.5

        res = math.floor(balanced_log_loss(y, preds)*100)/100

        score_dict[b] = res

    res = pd.Series(score_dict)
    return res

res = get_score_list(tp, 0)

# left is combination of objects where 0 means that object is not in LB and 1 means that object is in LB 
# right is corresponding LB score
res

- offset 0: score is 1.29, that means that elements 0, 1 and 5 are in LB and have class label 1

- offset 6: score is 1.23, that means that element 11 is in LB and has class label 1

- offset 12: score is 0.7, that means that element 12 is in LB and has class label 1

- offset 18: 

- offset 24:

In [ ]:
offset = 78

# sort predictions of model
_p = submission.copy()
_p.loc[:,'class_1'] = 1 - probabilities[:,:1]
_p = _p.sort_values('class_1', ascending=False) # change to ascending=True to find objects of class 0

# set predictions to special values
p_offset_len = _p['class_1'].iloc[offset:offset+len(tp)].shape[0]
if p_offset_len > 0:
    _p['class_1'].iloc[offset:offset+len(tp)] = tp[:p_offset_len]
    _p['class_1'].iloc[:offset] = 0.5
    _p['class_1'].iloc[offset+len(tp):] = 0.5
else:
    _p['class_1'] = 0.5

# if model is not sure enough in it's predictions - don't use these predictions
# p0 = probabilities[:,:1].copy()
# for i in range(p0.shape[0]):
#     if 0.28 <= p0[i] <= 0.59:
#         _p.iat[i, _p.columns.get_loc('class_1')] = 0.5

# set predictions for class 0
_p['class_0'] = 1 - _p['class_1']

# save predictions to the dataframe
_p = _p.sort_index()
_p.to_csv('submission.csv', index=False)
_p